# 06B PHA (CGenFF) + enzyme in water, GROMACS

**Workflow:** Enzyme–PHA in water (the method used for the project's research simulations): 05B → 06B

**Before this notebook:** prepare a docked complex, see notebook 11. Run 05B for the PHA parameters.

**What this does:** turns a CHARMM-GUI Solution Builder download (CHARMM36m protein, CGenFF PHA, CHARMM TIP3P water, SOD/CLA ions) into a GROMACS run folder and checks it.

**You get:** a checked run folder for 07. Without changes, the notebook uses the example data in `examples/data/charmm_gui_ANC55_P3HB4/` and writes to a temporary folder. GROMACS runs only if you set a `RUN_…` flag to `True`.

## Steps 2–4: build the system in CHARMM-GUI Solution Builder

**Inputs:** the docked complex PDB (PHA residue `LIG`, e.g. `Anc55_PHB4_pose2_complex.pdb`) and the 05B files. Upload `lig_g.rtf` as topology and `lig.prm` as parameters. If your download has no `lig_g.rtf`, use `lig.rtf`; the charges are identical.

| Page | Setting |
| --- | --- |
| PDB Info: Model/Chain Selection | tick **Protein** (`PROA`) and **Hetero** (`HETA`, `LIG`); tick **Check pKa** |
| PDB Manipulation | **System pH 7.0**; Reading Hetero Chain Residues: **Upload CHARMM top & par for hetero chain** → Topology `lig_g.rtf` (or `lig.rtf`), Parameter `lig.prm`; *Protonate/Deprotonate based on selected pH* ticked; terminal groups NTER/CTER (default) |
| Solvator: water box | **Fit Waterbox Size to Protein Size**, **Rectangular**, **edge distance 30 Å** |
| Solvator: ions | **Include ions**, ion placing **Monte-Carlo**, **NaCl 0.05 M**, neutralising; click *Calculate Solvent Composition* |
| Temperature | **303.15 K** |
| Input Generator | **GROMACS**; continue through the remaining pages and download the package |

Download and unpack the package; this notebook uses its `gromacs/` and `lig/` folders.

## Step 5: build the HPC folder

The notebook copies the download's `gromacs/` folder into a new run folder (the download is never changed) and renames `step3_input.gro` to `step5_input.gro`. It leaves out CHARMM-GUI's mdp files and adds the run files from `src/iphasimulator/data/charmm_gromacs/`: the `step6.0`–`step7` mdp files, `run_step6_local.sh` (local minimisation) and `run_hpc_equilibration_production.slurm` (steps 6.1, 6.2 and 7 on the cluster).

The stage lengths are printed from the mdp files, and the SLURM job name is the run folder's name. Set `TEMPLATE_DIR` to use your own run files.

## 1. Inputs

- `SOLUTION_BUILDER_DIR`: the unpacked Solution Builder download (it is only read). `None` uses the example data.
- `RUN_DIR`: a new folder for the run; its name becomes the SLURM job name. `None` makes a temporary folder. It may not be inside `examples/data/`.
- `SDF_PATH`: the SDF you uploaded in 05B, for the stereocentre check.
- `LIG_RTF`: the rtf for the charge check. `None` takes `lig_g.rtf`, or `lig.rtf` if there is none, from the download's `lig/` folder.
- `TEMPLATE_DIR`: `None` for the packaged run files, or your own folder.

In [ ]:
from pathlib import Path
import sys
import tempfile

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))
from iphasimulator.charmmgui_import import (
    check_atom_counts,
    check_cgenff_types,
    check_cgenff_penalties,
    check_charmm_defaults,
    check_gromacs_run_files,
    check_includes,
    check_ligand_atom_order,
    check_ligand_charges_match_rtf,
    check_ligand_signed_volumes,
    check_molecule_counts,
    check_net_charge,
    describe_run_folder_protocol,
    find_lig_topology,
    format_checks,
    prepare_charmm_gui_run_folder,
    read_minimization_result,
    read_topology_defaults,
    read_topology_molecule_counts,
)
from iphasimulator.simulation_gromacs_runner import run_gromacs_local_minimization

SOLUTION_BUILDER_DIR = None  # None: the example dataset; or e.g. Path("/path/to/Anc55_PHB4-XXXXXXXXXX")
RUN_DIR = None               # None: a new temporary folder; or e.g. Path("/path/to/new/ANC55_P3HB4_gromacs")
SDF_PATH = None              # e.g. repo_root / "examples/output/polymer_structures/PHB4_R.sdf"
LIG_RTF = None               # None: lig_g.rtf, or lig.rtf, from <SOLUTION_BUILDER_DIR>/lig/
TEMPLATE_DIR = None
JOB_NAME = None              # None: the RUN_DIR folder name

EXAMPLE_DATA_DIR = repo_root / "examples" / "data"
if SOLUTION_BUILDER_DIR is None:
    SOLUTION_BUILDER_DIR = EXAMPLE_DATA_DIR / "charmm_gui_ANC55_P3HB4"
    SDF_PATH = SDF_PATH or SOLUTION_BUILDER_DIR / "PHB4_R.sdf"
    print("Using the example dataset:", SOLUTION_BUILDER_DIR)
if RUN_DIR is None:
    name = "ANC55_P3HB4_gromacs" if Path(SOLUTION_BUILDER_DIR).parent == EXAMPLE_DATA_DIR else f"{Path(SOLUTION_BUILDER_DIR).name}_gromacs"
    RUN_DIR = Path(tempfile.mkdtemp(prefix="06B_")) / name
if EXAMPLE_DATA_DIR.resolve() in (Path(RUN_DIR).expanduser().resolve(), *Path(RUN_DIR).expanduser().resolve().parents):
    raise ValueError(f"RUN_DIR must not be inside {EXAMPLE_DATA_DIR}; choose another folder")
print("Run folder:", RUN_DIR)

if LIG_RTF is None and SOLUTION_BUILDER_DIR is not None:
    LIG_RTF = find_lig_topology(Path(SOLUTION_BUILDER_DIR) / "lig")  # lig_g.rtf, or lig.rtf if there is none

## 2. Prepare the run folder

In [ ]:
if SOLUTION_BUILDER_DIR is None or RUN_DIR is None:
    print("Set SOLUTION_BUILDER_DIR and RUN_DIR in step 1 to prepare the run folder.")
else:
    run_dir = prepare_charmm_gui_run_folder(
        SOLUTION_BUILDER_DIR, RUN_DIR, template_dir=TEMPLATE_DIR, job_name=JOB_NAME
    )
    print("Prepared:", run_dir)
    for path in sorted(run_dir.iterdir()):
        print("  ", path.name + ("/" if path.is_dir() else ""))
    print("\nStages (from the mdp files):")
    for stage in describe_run_folder_protocol(run_dir):
        restraints = "with position restraints" if stage.position_restraints else "no restraints"
        print(f"  {stage.name}: {stage.length_text()}, {restraints}")
    print("\nSLURM:")
    for line in (run_dir / "run_hpc_equilibration_production.slurm").read_text().splitlines():
        if line.startswith(("#SBATCH --job-name", "# Step")):
            print("  " + line)

## 3. Checks

- `[ defaults ]` from `toppar/forcefield.itp` is CHARMM-style (`1 2 yes 1.0 1.0`), and every `#include` file exists.
- Molecules: `PROA`, `LIG`, `SOD`, `CLA`, `TIP3`, and a net charge of about 0.
- CGenFF penalties in `lig/` (highest parameter / charge penalty): PASS below 10 (fair analogy), WARN 10–50 (basic validation recommended), FAIL above 50. Example: ANC55_P3HB4, 4.5 / 3.539.
- `LIG` charges and atom types equal the rtf (`lig_g.rtf`, or `lig.rtf`; the charges are identical); the atom types are CGenFF-style and in the same order as the coordinates.
- `step5_input.gro` has as many atoms as the topology.
- Every PHA stereocentre is still R: for each CG311 carbon, the signed volume of its O, CH2 and CH3 (or CH2) neighbours in `step5_input.gro` has the same sign as in the uploaded R SDF. Example: ANC55_P3HB4, 4/4 R.
- All run files are present and `index.ndx` has the `SOLU`/`SOLV` groups the mdp files use.

The second cell checks the system type: an enzyme + PHA package must contain a protein chain, the PHA (`LIG`), water and ions.

In [ ]:
if RUN_DIR is None or not Path(RUN_DIR).is_dir():
    print("Prepare the run folder in step 2 first.")
else:
    run_dir = Path(RUN_DIR)
    topology, gro = run_dir / "topol.top", run_dir / "step5_input.gro"
    print("[ defaults ] (toppar/forcefield.itp):", " ".join(read_topology_defaults(topology)))
    print("[ molecules ]:", ", ".join(f"{name} {count}" for name, count in read_topology_molecule_counts(topology).items()))
    print()
    print(format_checks([
        check_charmm_defaults(topology),
        check_includes(run_dir),
        check_molecule_counts(topology, require_water=True),
        check_net_charge(topology),
        check_cgenff_penalties(Path(LIG_RTF).parent if LIG_RTF else None),
        check_ligand_charges_match_rtf(topology, LIG_RTF),
        check_cgenff_types(topology),
        check_atom_counts(run_dir, "step5_input.gro"),
        check_ligand_atom_order(gro, topology),
        check_ligand_signed_volumes(gro, topology, SDF_PATH),
        check_gromacs_run_files(run_dir),
    ]))

In [ ]:
from iphasimulator.charmmgui_import import ION_MOLECULES, WATER_MOLECULES, read_topology_molecule_counts

if RUN_DIR is None or not (Path(RUN_DIR) / "topol.top").is_file():
    print("Prepare the run folder in step 2 first.")
else:
    counts = read_topology_molecule_counts(Path(RUN_DIR) / "topol.top")
    names = {name.upper(): count for name, count in counts.items()}
    protein = [name for name in counts if name.upper().startswith("PRO")]
    is_enzyme_polymer_in_water = (
        bool(protein)
        and names.get("LIG", 0) >= 1
        and any(names.get(water, 0) for water in WATER_MOLECULES)
        and any(names.get(ion, 0) for ion in ION_MOLECULES)
    )
    print(f"[{'PASS' if is_enzyme_polymer_in_water else 'FAIL'}] enzyme + polymer in water (protein + LIG + water + ions): {counts}")

## 4. Optional: local minimisation

Runs `bash run_step6_local.sh` in `RUN_DIR` (`grompp` + `mdrun` for `step6.0_minimization`, with the backbone/side-chain restraints from CHARMM-GUI). For an enzyme–polymer box of about 140 000 atoms this takes minutes on a laptop.

In [ ]:
RUN_LOCAL_MINIMIZATION = False

if not RUN_LOCAL_MINIMIZATION:
    print("Set RUN_LOCAL_MINIMIZATION = True to run step 6.0 locally.")
else:
    run_gromacs_local_minimization(RUN_DIR)
    result = read_minimization_result(Path(RUN_DIR) / "step6.0_minimization.log")
    print(f"Potential energy: {result.potential_energy} kJ/mol")
    print(f"Maximum force:    {result.maximum_force} kJ/mol/nm")

## 5. Hand-off to 07

`07_hpc_execution.ipynb` (CHARMM/GROMACS section) needs the run folder with `step6.0_minimization.gro` from step 4 and `run_hpc_equilibration_production.slurm`, which runs NVT (6.1), NPT (6.2) and production (7) on the cluster. The SLURM module line is for the KCL GPU cluster; adapt it for other systems.

In [ ]:
if RUN_DIR is None:
    print("No run folder yet.")
else:
    run_dir = Path(RUN_DIR)
    for name in ("step6.0_minimization.gro", "run_hpc_equilibration_production.slurm", "index.ndx", "topol.top"):
        print(f"{'found  ' if (run_dir / name).exists() else 'missing'}  {name}")
    print("On the cluster:")
    print(f"  cd {run_dir}")
    print("  sbatch run_hpc_equilibration_production.slurm")